# 09. Nested, Ragged & Specialized Sparse Tensors

### Historical Lineage & Theoretical Foundations
- **Lineage:** *PyTorch 2 NestedTensor & Dispatch Team (2023); Saad (2003, Iterative Methods for Sparse Linear Systems); Tinney & Walker (1967, Sparse Matrix Direct Methods)*.
- **Core Purpose:** High-performance processing of variable-length sequence batches without rectangular padding waste, contiguous memory layouts via offset pointer vectors, and dual equivalence to Compressed Sparse Row (CSR) SpMV.

### Decision Boundary: When NOT to Use
> [!WARNING]
> - **DO NOT use sparse CSR/COO formats when density exceeds 20%:** Indirect pointer indexing and non-vectorized memory gather overhead degrade latency compared to dense BLAS GEMMs at moderate/high density.
> - **DO NOT pad highly skewed sequence batches to maximum length:** Rectangular batches allocate $\mathcal{O}(B \cdot L_{\max})$ space, wasting memory bandwidth and FLOPs on padded zeros.
> - **DO NOT perform non-vectorized Python loops over batch slices in production:** Use contiguous ragged flattening or fused kernel gather/scatter reductions.


### Mathematical Formulation & Asymptotic Complexity

#### 1. Ragged Sequence Memory Mapping
$$\mathbf{X} \in \mathbb{R}^{N \times D}, \quad N = \sum_{i=1}^B L_i, \quad \mathbf{O} = [O_0, O_1, \dots, O_B] \in \mathbb{N}^{B+1}, \quad O_0 = 0, \ O_i = \sum_{k=1}^i L_k$$

#### 2. Ragged Mean Reduction Formulation
$$\mathbf{p}_i = \frac{1}{O_{i+1} - O_i} \sum_{t = O_i}^{O_{i+1} - 1} \mathbf{y}_t \in \mathbb{R}^{D_{\text{out}}}$$

#### 3. CSR SpMV Dual Equivalence
$$(\mathbf{M}_{\text{CSR}} \mathbf{Y})_{i, d} = \sum_{k = \text{crow}[i]}^{\text{crow}[i+1] - 1} \text{val}[k] \cdot \mathbf{Y}_{\text{col}[k], d} \equiv \mathbf{p}_i$$

**Asymptotic Complexity:**
- Dense Padded Batch: Memory $\mathcal{O}(B \cdot L_{\max} \cdot D)$; Projection Time $\mathcal{O}(B \cdot L_{\max} \cdot D \cdot D_{\text{out}})$.
- Ragged Nested Batch: Memory $\mathcal{O}(N \cdot D + B)$; Projection Time $\mathcal{O}(N \cdot D \cdot D_{\text{out}})$, eliminating $(B \cdot L_{\max} - N) \cdot D$ padding waste.


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn

import core

env = core.setup_env(seed=42)
ckpt_path = Path("weights/00_foundations/09_nested_ragged_and_specialized_sparse_tensors.safetensors")
if not ckpt_path.is_file():
    ckpt_path = Path("../../weights/00_foundations/09_nested_ragged_and_specialized_sparse_tensors.safetensors")
weights, test_in, golden_out, meta = core.parity.load_reference_checkpoint(ckpt_path)
dim = int(meta.get("dim", 4))
total_tokens = int(meta.get("total_tokens", 9))
num_seqs = int(meta.get("num_seqs", 3))
print(f"Device: {env.device} | Sequences: {num_seqs} | Tokens: {total_tokens} | Dim: {dim}")


In [ ]:
def ragged_mean_pool(values: torch.Tensor, offsets: torch.Tensor) -> torch.Tensor:
    num_s = offsets.shape[0] - 1
    pooled = torch.zeros(num_s, values.shape[-1], dtype=values.dtype, device=values.device)
    for i in range(num_s):
        s, e = int(offsets[i].item()), int(offsets[i + 1].item())
        pooled[i] = torch.mean(values[s:e], dim=0)
    return pooled

class RaggedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.Parameter(torch.empty(out_features))

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return values @ self.weight + self.bias

    def forward_pooled(self, values: torch.Tensor, offsets: torch.Tensor) -> torch.Tensor:
        return ragged_mean_pool(self.forward(values), offsets)

model = RaggedLinear(dim, 2)
with torch.no_grad():
    model.weight.copy_(weights["weight"])
    model.bias.copy_(weights["bias"])
print("RaggedLinear initialized with reference parameters.")


In [ ]:
# 1. Forward Token Projection Parity vs Golden Checkpoint
token_pred = model(test_in)
torch.testing.assert_close(token_pred, golden_out, atol=1e-5, rtol=1e-5)

# 2. Ragged Mean Pooling Parity
pooled_pred = model.forward_pooled(test_in, weights["offsets"])
torch.testing.assert_close(pooled_pred, weights["pooled_out"], atol=1e-5, rtol=1e-5)

# 3. CSR SpMV Dual Equivalence Identity
crow = weights["csr_crow"]
col = weights["csr_col"]
val = weights["csr_val"]
csr_mat = torch.sparse_csr_tensor(crow, col, val, (num_seqs, total_tokens))
csr_pooled = torch.matmul(csr_mat, token_pred)
torch.testing.assert_close(csr_pooled, weights["pooled_out"], atol=1e-6, rtol=1e-6)

# 4. Autograd Gradient Parity
v_param = test_in.clone().detach().requires_grad_(True)
loss = 0.5 * torch.sum((model(v_param) - weights["target"]) ** 2)
loss.backward()
torch.testing.assert_close(model.weight.grad, weights["grad_weight"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(model.bias.grad, weights["grad_bias"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(v_param.grad, weights["grad_input"], atol=1e-5, rtol=1e-5)
print("All mathematical invariants (Tokens + CSR SpMV Pooling + Gradients) verified.")


In [ ]:
def run_ragged_gemm():
    return model(test_in)

def run_padded_gemm():
    return weights["dense_padded"] @ weights["weight"] + weights["bias"]

def run_csr_spmv():
    return torch.matmul(csr_mat, token_pred)

track_fns = {"Ragged Contiguous GEMM": run_ragged_gemm, "Dense Padded GEMM": run_padded_gemm, "CSR SpMV Pooling": run_csr_spmv}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=total_tokens)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# [!WARNING] PATHOLOGICAL MISUSE & FAILURE MODE: Rectangular Padding Memory Waste
# 1. Flawed Anti-Pattern: Padding ragged sequences to max_len wastes memory allocations
dense_elements = weights["dense_padded"].numel()
ragged_elements = test_in.numel()
waste_elements = dense_elements - ragged_elements

# Active assertion proving severe allocation overhead
assert waste_elements > 0, "Dense padding must allocate strictly more elements than ragged storage"
waste_pct = (waste_elements / dense_elements) * 100.0
print(f"[FAILURE DEMONSTRATED] Rectangular padding wasted {waste_elements} elements ({waste_pct:.1f}% overhead)!")

# 2. Correct Pattern: Ragged contiguous layout eliminates 100% of padding elements
assert test_in.shape[0] == total_tokens
print(f"[CORRECT] Ragged batch packed into exact {total_tokens} active tokens with zero wasted slots.")

# 3. Compilation Verification
ok_exp, msg_exp = core.export.verify_torch_export(model, test_in)
ok_comp, msg_comp = core.export.verify_torch_compile_fullgraph(model, test_in)
assert ok_exp and ok_comp, "Compilation / export failed"
print("[CORRECT] Model verified with torch.compile(fullgraph=True) and exported.")


## Architectural Topology, Key Takeaways & Curriculum Navigation

### Ragged Batch & CSR SpMV Topology
<svg width="650" height="170" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <defs>
    <marker id="arrow-r" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#38bdf8"/>
    </marker>
  </defs>
  <text x="30" y="25" fill="#94a3b8" font-size="12" font-weight="bold">RAGGED JAGGED TENSORS &amp; CSR SPMV DUAL EQUIVALENCE</text>
  <rect x="30" y="50" width="130" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="95" y="72" fill="#38bdf8" font-size="11" text-anchor="middle">Ragged Values [N, D]</text>
  <line x1="160" y1="67" x2="220" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-r)"/>
  <rect x="220" y="50" width="170" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="305" y="72" fill="#10b981" font-size="11" text-anchor="middle">Offset Pointers [B+1]</text>
  <line x1="390" y1="67" x2="450" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-r)"/>
  <rect x="450" y="50" width="160" height="35" rx="6" fill="#1e293b" stroke="#a855f7" stroke-width="1.5"/>
  <text x="530" y="72" fill="#a855f7" font-size="11" text-anchor="middle">CSR SpMV / Pooling</text>
  <rect x="30" y="110" width="580" height="35" rx="6" fill="#0f172a" stroke="#f59e0b" stroke-width="1.5" stroke-dasharray="4 4"/>
  <text x="320" y="132" fill="#f59e0b" font-size="11" text-anchor="middle">Dual Isomorphism: Y_pooled == M_CSR @ Y_tokens (Zero-Padding Bandwidth Saved)</text>
</svg>

### Key Takeaways
1. **Zero Padding Tax:** Ragged tensors eliminate quadratic and rectangular memory waste by concatenating active tokens into contiguous cache-line memory.
2. **CSR SpMV Equivalence:** Per-sequence pooling over offsets is mathematically identical to sparse matrix multiplication against an incidence matrix.
3. **Dense BLAS Compatibility:** Flattening ragged tokens allows standard high-throughput GEMM calls across all tokens without custom kernel dispatch.

---
**Curriculum Navigation:**  
&larr; [Previous: 08. Reproducible Preprocessing Pipelines & Determinism](08_reproducible_preprocessing_pipelines_and_determinism.ipynb) | [Master Index: README.md](../../README.md) | [Next: 10. Floating-Point Arithmetic, Conditioning & Numerical Stability](10_floating_point_arithmetic_conditioning_and_numerical_stability.ipynb) &rarr;
